# SasRec — RePlay

Лучшая модель роли 2. Один запуск блокнота: проверка Recall@10, жанры, обучение на всех данных, top-100 и CSV для Kaggle.

Лучший отдельный запуск без жанров: **0.288411**. В последнем компактном запуске с весом жанров 0.10: **0.287583**. Вес выбирается по локальной валидации.

Запускать из корня репозитория `rexis-main` с GPU. Нужные пакеты: `pip install -r requirements-role2.txt`.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import lightning as L
from replay.preprocessing import LabelEncoder, LabelEncodingRule
from replay.data.nn.utils import groupby_sequences
from replay.data import FeatureHint, FeatureType
from replay.data.nn import TensorFeatureInfo, TensorSchema, ParquetModule
from replay.nn.transform.template import make_default_sasrec_transforms
from replay.nn.sequential import SasRec
from replay.nn.lightning import LightningModule
from replay.nn.lightning.callback import PandasTopItemsCallback

if not Path('data/events.csv').exists() and Path('/content/rexis-main/data/events.csv').exists():
    os.chdir('/content/rexis-main')
assert Path('data/events.csv').exists(), 'Запусти блокнот из папки rexis-main'
assert torch.cuda.is_available(), 'Для SasRec включи GPU в Colab'
from src.validation import leave_last_out
from src.metrics import recall_at_k_frame
from src.transformer_utils import decode_predictions, fill_top100, save_submission, genre_results, rerank_genres

EPOCHS = 30
SEQ_LEN = 100
DIM = 128
BLOCKS = 3
HEADS = 4
DROPOUT = 0.3
BATCH = 32
SEED = 42

L.seed_everything(SEED, workers=True)
events = pd.read_csv('data/events.csv')
items = pd.read_csv('data/item_features.csv')
sample = pd.read_csv('data/submission_sample.csv')
train, truth = leave_last_out(events)
Path('outputs').mkdir(exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| пользователей:', len(sample))

## Обучение и рекомендации

In [ ]:
def train_and_predict(history, name):
    encoder = LabelEncoder([LabelEncodingRule('user_id'), LabelEncodingRule('item_id')])
    encoded = encoder.fit_transform(history.copy())
    count = encoded.item_id.nunique()
    seq = groupby_sequences(encoded, groupby_col='user_id', sort_col='timestamp')
    temp = Path('outputs') / f'_{name}_tmp'
    temp.mkdir(exist_ok=True)
    seq.to_parquet(temp / 'train.parquet')
    seq.to_parquet(temp / 'predict.parquet')

    schema = TensorSchema([TensorFeatureInfo(
        name='item_id', is_seq=True, padding_value=count,
        cardinality=count, embedding_dim=DIM,
        feature_type=FeatureType.CATEGORICAL, feature_hint=FeatureHint.ITEM_ID
    )])
    transforms = make_default_sasrec_transforms(schema)
    train_data = ParquetModule(
        train_path=str(temp / 'train.parquet'), batch_size=BATCH,
        metadata={'train': {'item_id': {'shape': SEQ_LEN + 1, 'padding': count}}},
        transforms=transforms
    )
    net = SasRec.from_params(
        schema=schema, embedding_dim=DIM, max_sequence_length=SEQ_LEN,
        num_heads=HEADS, num_blocks=BLOCKS, dropout=DROPOUT
    )
    model = LightningModule(net)
    trainer = L.Trainer(
        max_epochs=EPOCHS, accelerator='gpu', devices=1,
        logger=False, enable_progress_bar=False, enable_model_summary=False,
        limit_val_batches=0, default_root_dir=str(temp)
    )
    trainer.fit(model, datamodule=train_data)

    prediction_data = ParquetModule(
        predict_path=str(temp / 'predict.parquet'), batch_size=BATCH,
        metadata={'predict': {'user_id': {}, 'item_id': {'shape': SEQ_LEN, 'padding': count}}},
        transforms=transforms
    )
    callback = PandasTopItemsCallback(
        top_k=min(200, count), query_column='user_id',
        item_column='item_id', rating_column='score'
    )
    predictor = L.Trainer(
        accelerator='gpu', devices=1, callbacks=[callback],
        logger=False, enable_progress_bar=False
    )
    predictor.predict(model, datamodule=prediction_data, return_predictions=False)
    recs = decode_predictions(callback.get_result(), encoder, history)
    recs = fill_top100(recs, history, sample.user_id, items.item_id)
    assert recs.groupby('user_id').size().eq(100).all()
    assert not pd.MultiIndex.from_frame(recs[['user_id', 'item_id']]).isin(
        pd.MultiIndex.from_frame(history[['user_id', 'item_id']])
    ).any()
    recs.to_csv(f'outputs/{name}_scores.csv', index=False)
    print(f'{name}: {len(recs)} рекомендаций')
    return recs

## Валидация и проверка жанров

In [ ]:
print('Обучаем SasRec для валидации')
val_raw = train_and_predict(train, 'sasrec')
by_genre = genre_results(val_raw, train, truth, items)
by_genre.to_csv('outputs/genre_experiments.csv', index=False)
print(by_genre.to_string(index=False))

# Выбираем вес по общей валидации
best = by_genre.sort_values(['recall@10', 'genre_weight'], ascending=[False, True]).iloc[0]
best_weight = float(best['genre_weight'])
recall = float(best['recall@10'])
val_recs = rerank_genres(val_raw, train, items, best_weight)
val_recs.to_csv('outputs/sasrec_scores.csv', index=False)
print(f'Лучший вес жанров: {best_weight:.2f}, Recall@10: {recall:.6f}')

## Предсказания для команды и Kaggle

In [ ]:
print('Обучаем SasRec на всей истории')
full_raw = train_and_predict(events, 'sasrec_full')
full_recs = rerank_genres(full_raw, events, items, best_weight)
full_recs.to_csv('outputs/sasrec_full_scores.csv', index=False)
submission = save_submission(full_recs, sample, 'outputs/submission_sasrec.csv')
assert submission.shape == sample.shape
assert submission.item_id.str.split().str.len().eq(10).all()

from zipfile import ZipFile, ZIP_DEFLATED
summary = pd.DataFrame([{
    'model': 'sasrec', 'recall@10': recall,
    'genre_weight': best_weight, 'users': len(sample)
}])
summary.to_csv('outputs/role2_summary.csv', index=False)
with ZipFile('outputs/role2_results.zip', 'w', compression=ZIP_DEFLATED) as archive:
    for name in ['sasrec_scores.csv', 'sasrec_full_scores.csv', 'submission_sasrec.csv',
                 'genre_experiments.csv', 'role2_summary.csv']:
        archive.write(f'outputs/{name}', arcname=name)
print('Готово: outputs/role2_results.zip')